# Mapping TissueTag annotations to traget object

### This notebook illustrates how to migrate TissueTag annotations to visium spot space in the form of an AnnData object but can also be used to match annotations to any type of spatial data in the form of a pandas DF

1) We will load the TissueTag annotated image of the visium dataset and translate the pixel level annotations to an hexagonal binned grid.
2) we will measure the minimal euclidean distances to the annotations and calculate 2 types of biological axes(OrganAxis). 
3) Finally we will migrate annotations to visium space and print some plots.

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import figure
import tissue_tag as tt 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

# Part 1 - Load annotation names and colors 

In [ ]:
# set path
path_to_tissue_tag = ".."
path = path_to_tissue_tag + '/data/tissue_tag_minimal_example_visium/' 

In [ ]:
from PIL import Image

# load tissue annotations from TissueTag and migrate annotation to a15um hexagonal grid and outputs the coordinates in ppm=1
tt_object = tt.load_annotation(file_path=path + '/tissue_annotations/annotations.h5')

# Use existing annotation from Nadav
tt_object.label_image = np.array(Image.open(path + '/tissue_annotations/annotations.tif'))

tt_object.grid = tt.generate_grid_from_annotation(tt_object, grid_unit_size= 15)
tt_object.grid.info()

In [ ]:
tt_object.grid['annotation'].value_counts()

# Part 2 - Calculate distances to structures and 2 versions of the axis

Here we calculate the relative distances of each spot in the grid to the closest corresponding K spots for each of the annotation categories. 

*note that here I selected K=10 to match the way we calculate the Cortico-Medullary axis for the thymus. However, since we have both broad (cortex,medulla) and fine grained (HS and PVS) structures in the same annotation. This might not be ideal for the fine grained structures. 

Ideally, you might prefer to have multiple annotation "layers" where you have broad and fine annotations separately as we have done in the paper and then select different K for each.   

In [ ]:
print('calculating distances')
tt_object.grid = tt.calculate_distance_to_annotations(
    grid_df=tt_object.grid,
    knn=2
    ) # calculate minimum mean distance of each spot to clusters

### Calculate morphological axes 

In [ ]:
# axis calculations based on 3 landmarks  - this axis is good to capture the spatial variance across the entire thymus 
structure = ['L2_dist_annotation_Edge','L2_dist_annotation_Cortex','L2_dist_annotation_Medulla']
w = [0.2,0.8]
tt_object.grid = tt.calculate_axis(tt_object.grid, feature_columns=structure, output_column='cma_3p', weights=w)

# axis calculations based on 2 landmarks - this axis is good to capture the location within the cortex
structure = ['L2_dist_annotation_Edge','L2_dist_annotation_Medulla']
tt_object.grid = tt.calculate_axis(tt_object.grid, feature_columns=structure, output_column='cma_2p')

In [ ]:
# remove non-annotated spots this is better to do after axis calculations
tt_object.grid = tt_object.grid[tt_object.grid['annotation']!='unassigned']
tt_object.grid = tt_object.grid[tt_object.grid['annotation']!='Artifacts']

# Part 3 - map annotations to visium spots via spatial KNN and plot some outputs

In [ ]:
# read visium info and map annotations  - make sure the 2 grids seem aligned
#_,ppm_vis,vis_df = tt.read_visium(spaceranger_dir_path=path + '/', plot=False)

tt_object.positions.rename(columns={'pxl_col': "x", 'pxl_row': "y"},inplace=True) # for new spaceranger adjust for any DF not only for visium
vis_df = tt.map_annotations_to_target(
    df_target=tt_object.positions,
    df_source=tt_object.grid,
    ppm_target= tt_object.ppm,
    plot=True,
)

In [ ]:
# read visium full data 
import scanpy as sc
adata_vis = sc.read_visium(path, count_file='raw_feature_bc_matrix.h5')

In [ ]:

# Align indices between adata_vis.obs and df_visium_spot
df_visium_spot_aligned = vis_df.iloc[:, 3:].reindex(adata_vis.obs.index)
adata_vis.obs = pd.concat([adata_vis.obs,vis_df.iloc[:,2:]],axis=1)
adata_vis.obsm['spatial'] = adata_vis.obsm['spatial'].astype('int')
adata_vis

In [ ]:
# plot the newly annotated visium AnnData
sc.set_figure_params(figsize=[10,10],dpi=75)
sc.pl.spatial(adata_vis,color=['annotation','L2_dist_annotation_Cortex', 'L2_dist_annotation_Edge', 'L2_dist_annotation_HS', 'L2_dist_annotation_Medulla', 'cma_3p', 'cma_2p']
              ,cmap='gist_rainbow',ncols=3)